# Chạy code gốc của tác giả (`upstream/run_model.py`) trên đúng 300 câu

Mục đích: có kết quả **tham chiếu** từ code gốc, chạy trên **cùng 300 câu** (`data/subset_300_ids.txt`) với notebook chính `adaptive_retrieval_repro.ipynb`, để đối chiếu ở issue #10 và #11. Nếu hai bên chạy hai tập câu khác nhau thì độ chính xác chênh nhau có thể chỉ do bốc mẫu, không phải do code.

Notebook này **không sửa** `run_model.py`. Nó làm lần lượt: cài thư viện còn thiếu, chép `upstream/` ra thư mục làm việc, gọi `run_model.py` cho `vanilla` và `BM25` với GPT-Neo 1.3B, rồi **kiểm id của kết quả có khớp đúng 300 id trong `data/subset_300_ids.txt` không**. Chỉ khi khớp mới in độ chính xác; không khớp thì dừng và báo lỗi, đừng dùng số đó để đối chứng.

**Trước khi chạy (Kaggle):** Settings → Accelerator = GPU (T4 ×2 hoặc P100), Internet = On. Add Input → dataset BM25 (`popqa-bm25-retrieval`, có file `wikidata_bm25_14282_retrieved_docs.jsonl`). Repo phải public thì notebook mới clone được.

### Những chỗ phải lách (không sửa code gốc)
1. `requirements.txt` của tác giả không ghi `jsonlines` và `openai` dù code import hai thư viện này, nên cài thêm (`upstream/KNOWN_ISSUES.md` mục 9).
2. README tác giả ghi `--ret_file`, nhưng code nhận `--ret_path`. Dùng `--ret_path` (mục 6).
3. Code gốc chọn câu hỏi bằng `knowledge.sample(n=300)` với bộ sinh số ngẫu nhiên toàn cục đã đặt seed 633. Với seed này nó chọn đúng 300 id của `subset_300_ids.txt`. Vì vậy ta đưa vào file PopQA **đầy đủ** kèm `--sample 300`, thay vì lọc file xuống 300 dòng: nhờ thế 15 ví dụ mẫu vẫn được bốc trong cả 14.267 câu như paper. Ô kiểm ở mục 5 xác nhận điều này ở mỗi lần chạy, vì kết quả có thể đổi nếu phiên bản thư viện làm tiêu thụ bộ sinh số ngẫu nhiên trước bước chọn.
4. Code gốc ghi kết quả vào `results/` theo thư mục đang chạy. Ta chạy trong một bản sao của `upstream/`, nên thư mục trong repo không bị đổi.

### Giữ nguyên hành vi gốc (chỉ ghi lại, không lách)
- Ví dụ mẫu của BM25 không kèm đoạn văn (mục 1 của `KNOWN_ISSUES.md`). Quyết định có sửa hay không: issue #12.
- Mô hình nạp ở fp32 (mục 10). GPT-Neo 1.3B khoảng 5 GB nên vẫn vừa T4.
- Ví dụ mẫu chọn ngẫu nhiên mỗi lần chạy, còn notebook chính cố định theo `id`. Vì vậy hai bên không thể giống 100% từng câu; issue #11 sẽ so.


In [ ]:
%pip install -q jsonlines openai

## 1. Cấu hình

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

MODEL_NAME = "EleutherAI/gpt-neo-1.3B"
METHODS    = ["vanilla", "BM25"]   # code gốc còn có "genread"; BM25 cần file retrieval
N_SAMPLE   = 300                   # phải bằng số id trong data/subset_300_ids.txt
N_EXAMPLES = 15                    # số ví dụ mẫu (few-shot), như paper
DEVICE     = "cuda"                # "cpu" chỉ để thử nhỏ; 1.3B trên CPU rất chậm

REPO_URL = "https://github.com/vanductan-NLT/CS221.git"   # phải là repo public thì Kaggle mới clone được
REPO_REF = "main"                  # nhánh hoặc tag; PR chưa merge thì đặt tên nhánh của PR
RET_PATH = None                    # file BM25 .jsonl; None = tự dò trong /kaggle/input

IN_KAGGLE = Path("/kaggle/working").exists()
BASE    = Path("/kaggle/working") if IN_KAGGLE else Path("./upstream_kaggle_work")
REPO_DIR = BASE / "CS221"            # bản clone của repo
WORKDIR  = BASE / "upstream_run"     # bản sao của upstream/ để chạy; kết quả ghi vào WORKDIR/results
OUT_DIR  = BASE / "upstream_results" # kết quả giữ lại (Kaggle: chỉ giữ khi Save Version)

In [ ]:
import torch, transformers, pandas as pd
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__, "| pandas", pd.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(không có GPU)")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU: Settings -> Accelerator -> GPU, rồi chạy lại từ ô đầu.")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

## 2. Lấy repo và chép `upstream/` ra thư mục làm việc

In [ ]:
def sh(cmd, cwd=None):
    """Chạy lệnh và in output ngay khi có (tqdm của code gốc dùng \r nên in nguyên văn)."""
    print("$", " ".join(map(str, cmd)))
    p = subprocess.Popen(cmd, cwd=cwd, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    if code != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {code}): {' '.join(map(str, cmd))}")

if not (REPO_DIR / "upstream" / "run_model.py").exists():
    BASE.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "| commit", commit or "(không phải bản clone git)")

if WORKDIR.exists():
    if not (WORKDIR / "run_model.py").exists():
        raise RuntimeError(f"{WORKDIR} đã có nhưng không phải bản sao của upstream/; hãy đổi tên hoặc xóa rồi chạy lại.")
    shutil.rmtree(WORKDIR)   # bản sao do notebook này tạo ở lần chạy trước, làm lại cho sạch
shutil.copytree(REPO_DIR / "upstream", WORKDIR)
print("đã chép upstream/ ->", WORKDIR)

## 3. Kiểm dữ liệu đầu vào

In [ ]:
POPQA  = WORKDIR / "data" / "popQA.tsv"
SUBSET = REPO_DIR / "data" / "subset_300_ids.txt"
popqa = pd.read_csv(POPQA, sep="\t")
want_ids = [int(x) for x in SUBSET.read_text(encoding="utf-8").split()]
assert len(popqa) == 14267, f"popQA.tsv phải có 14267 dòng, thấy {len(popqa)}"
assert len(want_ids) == len(set(want_ids)) == N_SAMPLE, f"{SUBSET.name} phải có đúng {N_SAMPLE} id khác nhau"
assert set(want_ids) <= set(popqa["id"]), "subset có id không nằm trong popQA.tsv"
print(f"popQA.tsv: {len(popqa)} dòng | subset: {len(want_ids)} id")

if "BM25" in METHODS and RET_PATH is None:
    cands = []
    if Path("/kaggle/input").exists():
        cands = [p for p in Path("/kaggle/input").rglob("*.jsonl")
                 if "bm25" in p.name.lower() and not p.name.startswith(".") and "__MACOSX" not in p.parts]
    if not cands:
        raise FileNotFoundError("Không thấy file BM25 .jsonl trong /kaggle/input. Add Input dataset BM25, hoặc đặt RET_PATH.")
    RET_PATH = sorted(cands)[0]
print("BM25:", RET_PATH)

## 4. Chạy `run_model.py` (code gốc, không sửa)
Mỗi cách chạy một lần, cùng `--sample 300` và cùng file PopQA đầy đủ. `vanilla` nhanh; `BM25` chậm hơn nhiều vì mỗi ví dụ mẫu kèm đoạn văn nên prompt rất dài (bị cắt từ bên trái ở 2048 token, giống paper). Code gốc sinh từng câu một nên chạy 300 câu cũng mất một lúc.

In [ ]:
ALIAS = "popQA"

def result_path(method):
    name = MODEL_NAME.replace("/", "_")
    return WORKDIR / "results" / f"model={name}-input={ALIAS}-method={method}-shots={N_EXAMPLES}-n={N_SAMPLE}.csv"

timings = {}
for method in METHODS:
    cmd = [sys.executable, "run_model.py", "--model_name", MODEL_NAME, "--input_file", "data/popQA.tsv",
           "--alias", ALIAS, "--eval_method", method, "--n_examples", str(N_EXAMPLES),
           "--sample", str(N_SAMPLE), "--device", DEVICE]
    if method == "BM25":
        cmd += ["--ret_path", str(RET_PATH)]
    t0 = time.time()
    sh(cmd, cwd=WORKDIR)
    timings[method] = time.time() - t0
    print(f"\n=== {method}: xong sau {timings[method] / 60:.1f} phút ===\n")

## 5. Kiểm: kết quả có đúng 300 câu của `subset_300_ids.txt` không?
Đây là cổng kiểm. Với mỗi cách, id trong file kết quả phải **trùng đúng tập** 300 id (không thừa, không thiếu, không lặp). Không khớp thì dừng ngay, không in độ chính xác. Thứ tự chỉ được báo thêm: lệch thứ tự mà đúng tập thì vẫn so được theo `id`.

In [ ]:
checks, results = [], {}
for method in METHODS:
    path = result_path(method)
    if not path.exists():
        raise FileNotFoundError(f"Không thấy file kết quả: {path}")
    res = pd.read_csv(path)
    got = res["id"].astype(int).tolist()
    missing, extra = sorted(set(want_ids) - set(got)), sorted(set(got) - set(want_ids))
    exact = (not missing) and (not extra) and len(got) == len(set(got)) == N_SAMPLE
    checks.append(dict(method=method, n=len(got), khop_tap_id=exact, cung_thu_tu=(got == want_ids),
                       thieu=len(missing), thua=len(extra)))
    results[method] = res
check_df = pd.DataFrame(checks)
print(check_df.to_string(index=False))
if not check_df["khop_tap_id"].all():
    raise AssertionError(
        "KẾT QUẢ KHÔNG KHỚP data/subset_300_ids.txt, đừng dùng để đối chứng.\n"
        "Nguyên nhân thường gặp: phiên bản transformers/numpy/pandas làm tiêu thụ bộ sinh số ngẫu nhiên trước bước chọn câu, "
        "hoặc popQA.tsv khác bản trong repo. Xem cột thieu/thua ở bảng trên.")
print("\nĐÚNG: cả", len(METHODS), "kết quả khớp đúng", N_SAMPLE, "id của subset_300_ids.txt.")

rows = []
for method, res in results.items():
    row = dict(method=method, n=len(res), accuracy=round(float(res["is_correct"].mean()), 4),
               phut=round(timings.get(method, float("nan")) / 60, 1))
    if "has_answer" in res.columns:
        row["retrieval_co_dap_an"] = round(float(res["has_answer"].mean()), 4)   # đoạn BM25 có chứa đáp án
    rows.append(row)
summary = pd.DataFrame(rows)
print("\n" + summary.to_string(index=False))

## 6. Lưu kết quả

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for method in METHODS:
    shutil.copy(result_path(method), OUT_DIR / result_path(method).name)
print("Đã chép kết quả vào", OUT_DIR)
print("Kaggle: bấm Save Version -> Save & Run All (Commit) thì thư mục này mới được giữ lại.\n")
print("Gợi ý ghi vào results/RUNS.md (điền ngày và link phiên bản notebook):")
print(f"  code gốc run_model.py | {MODEL_NAME} | {', '.join(METHODS)} | {N_SAMPLE} câu | {GPU_NAME} | "
      + " ; ".join(f"{m} {timings[m] / 60:.1f} phút" for m in METHODS)
      + " | " + " ; ".join(f"{r['method']} {r['accuracy'] * 100:.1f}%" for r in rows))